# EDA — F1 Top-10 Predictor

Análise exploratória do dataset final (`data/final/f1_dataset_final.parquet`).  
Granularidade: **1 linha por (raceId, driverId)** — resultado de corrida.  
Target: `top10` — 1 se o piloto terminou em posição ≤ 10, 0 caso contrário.

In [ ]:

# Detecta se está rodando no Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print("Faça upload do arquivo f1_dataset_final.parquet")
    uploaded = files.upload()
    DATA_PATH = 'f1_dataset_final.parquet'
else:
    DATA_PATH = '../data/final/f1_dataset_final.parquet'

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()


## 1. Distribuição do target `top10`

In [ ]:
counts = df['top10'].value_counts().sort_index()
labels = ['Fora do top10 (0)', 'Top10 (1)']
colors = ['#d9534f', '#5cb85c']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Contagem por classe', fontsize=13)
axes[0].set_ylabel('Corridas')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Proporção das classes', fontsize=13)

plt.suptitle('Distribuição do target top10', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Dataset balanceado: {counts[0]} fora ({counts[0]/len(df):.1%}) vs {counts[1]} top10 ({counts[1]/len(df):.1%})')

## 2. Taxa de top10 por posição de grid (largada)

In [ ]:
grid_valido = df[df['grid'].between(1, 20)].copy()
taxa_grid = grid_valido.groupby('grid')['top10'].mean().reset_index()
taxa_grid.columns = ['grid', 'taxa_top10']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(taxa_grid['grid'], taxa_grid['taxa_top10'],
              color=['#5cb85c' if t >= 0.5 else '#d9534f' for t in taxa_grid['taxa_top10']],
              edgecolor='white')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50%')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Posição de largada (grid)', fontsize=12)
ax.set_ylabel('Taxa de top10', fontsize=12)
ax.set_title('Taxa de chegada no top10 por posição de grid\n(verde = >50%, vermelho = <50%)', fontsize=13)
ax.set_xticks(taxa_grid['grid'])
ax.legend()
plt.tight_layout()
plt.show()
print('Conclusão: pilotos que largam nas primeiras 10 posições têm alta probabilidade de pontuar.')

## 3. Taxa de top10 por construtor

In [ ]:
taxa_constructor = (
    df.groupby('constructorId')['top10']
    .agg(taxa='mean', corridas='count')
    .reset_index()
    .sort_values('taxa', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#5cb85c' if t >= 0.5 else '#d9534f' for t in taxa_constructor['taxa']]
bars = ax.barh(taxa_constructor['constructorId'].astype(str),
               taxa_constructor['taxa'], color=colors, edgecolor='white')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

for bar, (_, row) in zip(bars, taxa_constructor.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['corridas']} corridas", va='center', fontsize=8, color='#555')

ax.set_xlabel('Taxa de top10', fontsize=12)
ax.set_ylabel('constructorId', fontsize=12)
ax.set_title('Taxa de chegada no top10 por construtor (2021–2023)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Distribuição das features climáticas

In [ ]:
clima_cols = ['TrackTemp', 'AirTemp', 'Humidity', 'Rainfall', 'WindSpeed']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(clima_cols):
    ax = axes[i]
    for top10_val, label, color in [(0, 'Fora top10', '#d9534f'), (1, 'Top10', '#5cb85c')]:
        subset = df[df['top10'] == top10_val][col]
        ax.hist(subset, bins=25, alpha=0.55, label=label, color=color, edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Valor médio na corrida')
    ax.set_ylabel('Frequência')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Distribuição das features climáticas por classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Nota: o clima é uma média das voltas da corrida — proxy das condições do dia.')

## 5. Heatmap de correlação das features numéricas

In [ ]:
num_cols = ['grid', 'round', 'year', 'TrackTemp', 'AirTemp', 'Humidity', 'Rainfall', 'WindSpeed', 'top10']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = __import__('numpy').triu(__import__('numpy').ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlação entre features numéricas e target', fontsize=13)
plt.tight_layout()
plt.show()
print('Destaque: grid tem correlação negativa com top10 — quanto menor o grid, maior a chance de pontuar.')

## 6. Taxa de top10 por circuito

In [ ]:
taxa_circuit = (
    df.groupby('circuitId')['top10']
    .agg(taxa='mean', corridas='count')
    .reset_index()
    .sort_values('taxa', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#5cb85c' if t >= 0.5 else '#d9534f' for t in taxa_circuit['taxa']]
ax.barh(taxa_circuit['circuitId'].astype(str), taxa_circuit['taxa'],
        color=colors, edgecolor='white')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='50%')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Taxa de top10', fontsize=12)
ax.set_ylabel('circuitId', fontsize=12)
ax.set_title('Taxa de top10 por circuito (2021–2023)\n(deve ser ~50% em todos — confirma ausência de leakage)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()
print('Se algum circuito tivesse taxa muito diferente de 50%, indicaria desbalanceamento ou leakage.')

## Resumo EDA

| Observação | Detalhe |
|------------|---------|
| Dataset balanceado | ~50/50 entre top10 e fora — sem necessidade de SMOTE |
| `grid` é a feature mais preditiva | Correlação negativa forte com `top10` |
| Construtor importa | Equipes top têm taxa de top10 muito acima de 50% |
| Clima tem baixa correlação individual | Mas pode interagir com grid e construtor |
| Ausência de leakage | Taxa de top10 ~50% em todos os circuitos e anos |